<br><div align="center"><span style="font-size:200%;font-weight:bold">Vectorisation des documents et des mots, vers l'inférence statistique</span></div></br>


<div align="center"><span style="font-size:150%">  Arnold Vialfont, UPEC - ERUDITE, UNIL </span></div>


<br>


<div align="center"><span style="font-size:130%">  Avril 2026</span></div>

***

Ce Notebook, vise à donner une idée des principales options existantes lors des étapes de **pre-processing** et de **feature engineering** du pipeline NLP. Il s'agit de transformer des textes bruts en vecteurs, après avoir "tokénisé" les chaînes de caractère et, si nécessaire, appliqué une normalisation des tokens obtenus. 

On verra d'abord l'approche de vectorisation par document puis des mots eux-mêmes.

Une première application est présentée pour illustrer l'étape de **modélisation**, en utilisant la régression logistique pour classer des documents (apprentissage automatique supervisé). 

Sommaire de ce notebook :

- **Tokénisation**
    - "à la main"
    - avec des packages existants
- **Normalisations**
    - retrait des stopwords et de la ponctuation
    - lemmatisation
    - stemming (racinisation)
- **Vectorisation**
    - encodage "one-hot" et de fréquence
    - encodage TF-IDF
    - Stockage des vecteurs en Dataframes
- **Reconnaissance des entités nommées (ou NER)**
- **Classification par régression logistique**
- **Références**

# 1. Tokénisation

Pour appliquer les algorithmes du NLP, il faut commencer par un **découpage des chaînes de caractères en éléments qui peuvent être analysés** (ayant ici un sens minimal). Ce découpage est appelé "tokénisation". Ce processus est parfois très simple voire invisible. Ce peut être une étape à paramétrer/entrainer explicitement, notamment en fonction de la langue du texte ou de la spécificité des thèmes abordés.

Il est habituel de réaliser cette tokénisation phrase par phrase, de sorte qu'une première étape est souvent la "**segmentation en phrase**" des textes. Dans certains cas, le **regroupement des phrases par paragraphe** peut avoir du sens, en particulier avec des textes longs et structurés de façon particulière (ex : textes juridiques).

La figure suivante illustre le processus de tokénisation au niveau des mots ("tokens") avec une phrase en anglais relativement simple.

<figure>
    <div align="center" style="margin-top:0.5cm"> <b> Tokénisation en anglais selon le site de la librairie <code>SpaCy</code> (<a href=https://spacy.io/usage/linguistic-features#tokenization>consulté en mars 2024</a>)</b><br>
  <img src="img/tokenizer_spacy.png" style="margin-top:0.5cm;margin-bottom:0cm;width:60%"></div>
<div align="center" style="margin-top:0.1cm"> RQ : "Prefix" se rapporte ici au caractère au début d'une chaîne (ex : “, ¿, (, $,) et "Suffix" à la fin (ex : ?, ), ”, !) </div>

</figure>


Pour cette étape de tokénisation d'un document, on peut :

- réaliser un découpage des chaînes de caractères "à la main" donc par expertise (avec les méthodes basiques des objets `string` sous python),

- utiliser un package dédié (solution la plus fréquente en pratique), ou

- entraîner un algorithme de tokenisation avec des exemples ($\simeq$ inférence grammaticale).


On se limitera ici deux premières solutions.

## Tokénisation à la main

Sous Python, les variables contenant des chaînes de caractères disposent de méthodes spécifiques qui pourrait permettre de réaliser cette étape. C'est en tout cas un bon moyen pour commencer une introduction au text mining.

- **Pour la tokénisation en "mots"**, il y a par exemple la méthode `split` qui crée une liste en coupant par défaut la chaîne à chaque espace (qui est supprimé durant l'opération) : 

In [10]:
#On définit une variable contenant du texte

string = '''Ce notebook est construit avec "Python 3.11" sous Windows ! Il peut être valable avec des versions de Python antérieures ou ultérieures. Cependant, rien d'assuré...'''

#L'utilisation de la fonction print n'est pas nécessaire mais condense l'affichage

print( string.split() )

['Ce', 'notebook', 'est', 'construit', 'avec', '"Python', '3.11"', 'sous', 'Windows', '!', 'Il', 'peut', 'être', 'valable', 'avec', 'des', 'versions', 'de', 'Python', 'antérieures', 'ou', 'ultérieures.', 'Cependant,', 'rien', "d'assuré..."]


<div class="alert alert-info">
<b> Info:</b> Pour déclarer une chaîne de caractères on encadre le texte de guillemets simples <code>'</code> s'il n'y a pas de caractère spécial dans la chaîne en question. On utilise des doubles guillemets <code>"</code> dans le cas contraire, voire trois guillemets simples <code>'''</code> (en particulier si la chaîne de caractères contient des doubles guillemets ou des retours à la ligne). </div>

Rappel : on peut regarder les options des méthodes associées à un objet avec la commande `help(object)`, où `object` est un type ou une méthode. On voit alors que dans la méthode `split` on peut spécifier tout séparateur :

In [13]:
#On peut choisir un sep='\t' pour les tabulations par exemple, et limiter le nombre de découpage avec maxsplit = 42
help(string.split)

Help on built-in function split:

split(sep=None, maxsplit=-1) method of builtins.str instance
    Return a list of the substrings in the string, using sep as the separator string.
    
      sep
        The separator used to split the string.
    
        When set to None (the default value), will split on any whitespace
        character (including \n \r \t \f and spaces) and will discard
        empty strings from the result.
      maxsplit
        Maximum number of splits.
        -1 (the default value) means no limit.
    
    Splitting starts at the front of the string and works to the end.
    
    Note, str.split() is mainly useful for data that has been intentionally
    delimited.  With natural text that includes punctuation, consider using
    the regular expression module.



- **Pour la segmentation en phrase des textes**, la méthode split peut aussi servir en utilisant certaines ponctuations (`.`, `!`, `?`, etc) :

In [15]:
#le séparateur est supprimé avec cette solution
print(string.split('.'))

['Ce notebook est construit avec "Python 3', '11" sous Windows ! Il peut être valable avec des versions de Python antérieures ou ultérieures', " Cependant, rien d'assuré", '', '', '']


On peut construire des règles en cascade pour tokéniser par phrase, mais le résultat est souvent insatisfaisant car il est presqu'impossible de définir des règles générales (ex : des `.` utilisés en dehors des fins de phrases ou des points de suspension) :

In [17]:
manual_tokens=[]

for elem in string.split('!'):
    
    for s in elem.split('.'):
    
        manual_tokens.append(s.split())

print(manual_tokens)

#On peut essayer avec des séparateurs intégrant un espace après la ponctuation : '. ' au lieu de '.'

[['Ce', 'notebook', 'est', 'construit', 'avec', '"Python', '3'], ['11"', 'sous', 'Windows'], ['Il', 'peut', 'être', 'valable', 'avec', 'des', 'versions', 'de', 'Python', 'antérieures', 'ou', 'ultérieures'], ['Cependant,', 'rien', "d'assuré"], [], [], []]


- **Enfin, on peut décomposer un texte en paragraphes**, ce qui peut être utile selon les cas d'usage. Cela peut être fait avec la méthode `split` en utilisant les caractères de saut de ligne (`\n`) ou de retour à la ligne (`\r`), voire les deux (`\n\r`). En cumlant les segmentation, le document est alors décomposé en listes imbriquées de la façon suivante :

                    [paragraphe1 [phrase1 [mot1 , mot2, ...], phrase2, ...] pragraphe2, ...]  

Cependant, on peut utiliser la méthode `splitlines` qui sépare les chaînes de caractère à n'importe lequel des signes <code>\r</code> et <code>\n</code> :

In [20]:
string2='''Attention : 
Ce notebook est construit \r avec Python 3.11 sous Windows ! \n Il peut être valable  \n\r avec des versions de Python antérieures ou ultérieures. Cependant, rien d'assuré...'''

#Lors de l'encodage des chaînes de caractères (en utf-8 par défaut), les retours à la ligne obtenus avec Enter deviennent des '\n'
string2

"Attention : \nCe notebook est construit \r avec Python 3.11 sous Windows ! \n Il peut être valable  \n\r avec des versions de Python antérieures ou ultérieures. Cependant, rien d'assuré..."

On trouve le résultat suivant :

In [22]:
string2.splitlines()

['Attention : ',
 'Ce notebook est construit ',
 ' avec Python 3.11 sous Windows ! ',
 ' Il peut être valable  ',
 '',
 " avec des versions de Python antérieures ou ultérieures. Cependant, rien d'assuré..."]

<div class="alert alert-danger">
<b> Attention :</b> Les sytèmes d'exploitation Windows utilisent uniquement <code>\n\r</code> ou  <code>\n</code> pour les retours et sauts à la ligne, Linux uniquement <code>\n</code> et Apple uniquement <code>\r</code>. Avec le module <code>print()</code> le résultat affiché peut ne pas être complet selon le système d'exploitation utilisé avec un <code>\r</code> ou un <code>\n</code> isolé.</div>

In [24]:
print(string2)

Attention : 
 avec Python 3.11 sous Windows ! 
 Il peut être valable  
 avec des versions de Python antérieures ou ultérieures. Cependant, rien d'assuré...


<div class="alert alert-info">
    <b> Info :</b> Le reste de ce notebook n'aborde plus la question du découpage en paragraphes, mais la librairie <code>NLTK</code> intègre une solution pour le faire. Par ailleurs, les <a href=https://fr.wikipedia.org/wiki/Expression_r%C3%A9guli%C3%A8re>expressions régulières</a> (ou <code>regex</code>) permettent des généralisations pour la tokénisation à la main, souvent insatisfaisantes par rapport aux approches automatisés avec des bibliothèques modernes.
</div>

## Tokénisation avec des packages existants

L'utilisation de librairies spécialisées est de loin la solution la plus fréquente pour la tokénisation. On se limitera ici à trois packages : 


- `NLTK` ([Site de référence](http://www.nltk.org/) et [Github](https://github.com/nltk/nltk)),


- `SpaCy` ([Site de référence](https://spacy.io/) et [Github](https://github.com/explosion/spaCy)), 


- `Scikit-Learn` ([Site de référence](https://scikit-learn.org/stable/) et [Github](https://github.com/scikit-learn/scikit-learn)).

Il peut être nécessaire de passer par une librairie plutôt qu'une autre selon votre projet ou l'environnement logiciel disponible. De plus, même si le plus simple est de choisir celleq ui intègre toutes les étapes du projet, ce n'est pas possible : il faut alors être attentif à la nature et/ou la forme des objets créés (listes de caractères, vecteurs de nombres reliés à un lexique, etc.).

D'autres packages existent pour une implémentation de la tokénisation et plus généralement du NLP sous Python mais ne seront pas utilisées ici :

- `Gensim` ([Site de référence](https://radimrehurek.com/gensim/) et [Github](https://github.com/RaRe-Technologies/gensim)) est une librairie pour le topic modeling.

-  `Transformers` de Hugging Face ([Site de référence](https://huggingface.co/) et [Github](https://github.com/huggingface)) pour certaines solutions SOTA.

- l'Université de Stanford développe plusieurs solutions dont `Stanza` ([Site de référence](https://stanfordnlp.github.io/) et [Github](https://github.com/stanfordnlp/stanza)). Ce package n'est pas utilisé ici.


### NLTK

Ce package est un des premiers à avoir été développé en matière de NLP. Le livre Bird et al. (2009) ([intégralement disponible ici](https://www.nltk.org/book)) présente ses principales fonctionnalités. De plus, plusieurs corpus de textes et ressources lexicales (très majoritairement en anglais) y sont intégrés et peuvent être téléchargés si besoin (cf. le chapitre 2 du livre).

On n'utilisera cette librairie que pour certaines tâches spécifiques pour lesquelles elle est nécessaires car elle est majoritairement prévu pour l'anglais : le français et autres langages ne sont disponibles que pour certaines des opérations de base (et la documentation n'est pas toujours très claire sur lesquelles). De plus, la librairie `SpaCy` est souvent plus rapide pour les mêmes tâches.

- **La tokénisation** est disponible en français en appliquant le module `word_tokenize` à une chaîne de caractères, et en spécifiant l'option `language='french'` (après avoir récupéré le tokéniseur `punkt`) :

In [31]:
#Après installation de cette librairie, il faut exécuter au moins une fois le code suivant (téléchargement de 2.5 Mo) :
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

#On importe alors le module word_tokenize
from nltk import word_tokenize

#https://fr.wikipedia.org/wiki/Traitement_automatique_des_langues
french_sent = "Le traitement automatique du langage naturel (abr. taln), ou traitement automatique de la langue naturelle, ou encore traitement automatique des langues (abr. TAL) est un domaine multidisciplinaire impliquant la linguistique, l'informatique et l'intelligence artificielle, qui vise à créer des outils de traitement de la langue naturelle pour diverses applications."

#L'option language permet la spécification de la langue, language='english' étant la valeur par défaut
print(word_tokenize(french_sent, language='french'))

['Le', 'traitement', 'automatique', 'du', 'langage', 'naturel', '(', 'abr', '.', 'taln', ')', ',', 'ou', 'traitement', 'automatique', 'de', 'la', 'langue', 'naturelle', ',', 'ou', 'encore', 'traitement', 'automatique', 'des', 'langues', '(', 'abr', '.', 'TAL', ')', 'est', 'un', 'domaine', 'multidisciplinaire', 'impliquant', 'la', 'linguistique', ',', "l'informatique", 'et', "l'intelligence", 'artificielle', ',', 'qui', 'vise', 'à', 'créer', 'des', 'outils', 'de', 'traitement', 'de', 'la', 'langue', 'naturelle', 'pour', 'diverses', 'applications', '.']


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\34550\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\34550\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


<div class="alert alert-info">
<b> Info :</b> <code>word_tokenize</code> applique un algorithme pré-entraîné, dont l'entrainement a été réalisé sur un corpus spécifique à chaque langue disponible. On peut trouver la liste des langues disponibles et les corpus utilisés sur son ordinateur, par exemple sous Windows dans le fichier <code>README</code> qui est dans le répertoire <code>C:\[user]\AppData\Roaming\nltk_data\tokenizers\punkt</code> et qu'on peut ouvrir par exemple avec <code>Notepad++</code>.  </div>

D'autres modules sont disponibles dans `NLTK` pour réaliser une tokénisation par `regex` : `wordpunct_tokenize` qui repose sur des regex prédéfinis et `regexp_tokenize` qui permet de les définir.

- **Pour la segmentation des phrases** (disponible pour le français), on peut utiliser le module `sent_tokenize` :

In [35]:
from nltk import sent_tokenize

#Un texte de plusieurs phrases de la même page Wikipédia
french_sents = "Le TALN est sorti des laboratoires de recherche pour être progressivement mis en œuvre dans des applications informatiques nécessitant l'intégration du langage humain à la machine. Aussi le TALN est-il parfois appelé ingénierie linguistique. En France, le traitement automatique de la langue naturelle a sa revue, Traitement automatique des langues, publiée par l’Association pour le traitement automatique des langues (ATALA)."

#L'option language='french' est disponible
sent_tokenize(french_sents, language='french')

["Le TALN est sorti des laboratoires de recherche pour être progressivement mis en œuvre dans des applications informatiques nécessitant l'intégration du langage humain à la machine.",
 'Aussi le TALN est-il parfois appelé ingénierie linguistique.',
 'En France, le traitement automatique de la langue naturelle a sa revue, Traitement automatique des langues, publiée par l’Association pour le traitement automatique des langues (ATALA).']

<div class="alert alert-info">
<b> Info :</b> <code>sent_tokenize</code> applique un algorithme qui a été entraîné sur les mêmes corpus que  <code>word_tokenize</code>.</div>

<div class="alert alert-danger">
<b> Attention :</b> Bien qu'entraînés sur des corpus importants, ces algorithmes et les suivants ne sont pas infaillibles. Il est important de bien vérifier les résultats qu'ils génèrent, notamment selon le type de texte (très littéraire, sms, etc.). Dans les exemples vus au-dessus, l'application de <code>sent_tokenize</code> au texte contenu dans <code>french_sent</code> segmente après les "abr.". Un traitement complémentaire à la main est toujours possible et souvent souhaitable !</div>

### SpaCy

`SpaCy` est une librairie relativement récente qui a été développée pour plusieurs langues ([voir cette page pour découvrir les langues disponibles et les fonctions, dans le menu latéral de gauche](https://spacy.io/models)). Ici, nous allons appliquer le modèle `fr_core_news_lg` qui pèse environ 570 Mo et a été entrainé pour plusieurs tâches du NLP sur un large corpus français. Il existe des versions de taille moyenne (`fr_core_news_md`) et petite (`fr_core_news_sm`), mais la plus volumineuse est plus précise. 

In [40]:
import spacy

#Avec une version de SpaCy >=3.0.0 on peut télécharger fr_core_news_lg, sinon seulement fr_core_news_md et/ou fr_core_news_sm)

#Etape de téléchargement nécessaire une seule fois (dans conda prompt) : python -m spacy download fr_core_news_lg

#Le chargement du modèle dans la RAM est un peu long 

nlp = spacy.load('fr_core_news_lg')

- **Pour la tokénisation**, on commence par appliquer l'objet `nlp` qu'on vient juste de créer à l'ensemble du texte, puis on peut utiliser la méthode `text` à chaque élément du `Doc` ainsi créé :

In [42]:
#Application à une seule phrase

doc_spacy1 = nlp(french_sent)

print("- SpaCy crée un objet appelé 'Doc' de type {}, qui peut directement être lu :\n".format(type(doc_spacy1)))

print(doc_spacy1, '\n')

print("- L'objet Doc est itérable, donc on peut créer une liste des tokens (avec la méthode .text qui retourne une chaîne de caractères) :\n")

print([token.text for token in doc_spacy1])

- SpaCy crée un objet appelé 'Doc' de type <class 'spacy.tokens.doc.Doc'>, qui peut directement être lu :

Le traitement automatique du langage naturel (abr. taln), ou traitement automatique de la langue naturelle, ou encore traitement automatique des langues (abr. TAL) est un domaine multidisciplinaire impliquant la linguistique, l'informatique et l'intelligence artificielle, qui vise à créer des outils de traitement de la langue naturelle pour diverses applications. 

- L'objet Doc est itérable, donc on peut créer une liste des tokens (avec la méthode .text qui retourne une chaîne de caractères) :

['Le', 'traitement', 'automatique', 'du', 'langage', 'naturel', '(', 'abr', '.', 'taln', ')', ',', 'ou', 'traitement', 'automatique', 'de', 'la', 'langue', 'naturelle', ',', 'ou', 'encore', 'traitement', 'automatique', 'des', 'langues', '(', 'abr', '.', 'TAL', ')', 'est', 'un', 'domaine', 'multidisciplinaire', 'impliquant', 'la', 'linguistique', ',', "l'", 'informatique', 'et', "l'", 'inte

In [43]:
#Equivalent à créer un boucle :
test=[]
for elem in doc_spacy1:
    test.append(elem.text)
print(test)

['Le', 'traitement', 'automatique', 'du', 'langage', 'naturel', '(', 'abr', '.', 'taln', ')', ',', 'ou', 'traitement', 'automatique', 'de', 'la', 'langue', 'naturelle', ',', 'ou', 'encore', 'traitement', 'automatique', 'des', 'langues', '(', 'abr', '.', 'TAL', ')', 'est', 'un', 'domaine', 'multidisciplinaire', 'impliquant', 'la', 'linguistique', ',', "l'", 'informatique', 'et', "l'", 'intelligence', 'artificielle', ',', 'qui', 'vise', 'à', 'créer', 'des', 'outils', 'de', 'traitement', 'de', 'la', 'langue', 'naturelle', 'pour', 'diverses', 'applications', '.']


- **Pour la segmentation des phrases**, chaque `Doc` créé intègre une méthode `.sents` (qui retourne des objets ayant une méthode `.text` qui renvoie la phrase sous forme d'une chaîne de caractères) :

In [45]:
#Application à un texte contenant plusieurs phrases

doc_spacy2 = nlp(french_sents)

[sent.text for sent in doc_spacy2.sents]

["Le TALN est sorti des laboratoires de recherche pour être progressivement mis en œuvre dans des applications informatiques nécessitant l'intégration du langage humain à la machine.",
 'Aussi le TALN est-il parfois appelé ingénierie linguistique.',
 'En France, le traitement automatique de la langue naturelle a sa revue, Traitement automatique des langues, publiée par l’Association pour le traitement automatique des langues (ATALA).']

### Scikit-Learn

Enfin, la librairie orientée machine learning `Scikit-Learn` permet aussi de réaliser du découpage de textes de façon automatisée. Cependant, elle est surtout utilisée pour son apport à l'étape de la modélisation et est tournée vers la création des vecteurs fournis aux algorithmes. Dans ce sens, elle réalise des traitements relativement (trop) basiques sur les textes. 

- **La tokénisation des mots** se fait même de façon implicite durant l'étape de vectorisation des textes, de sorte que pour obtenir la liste des tokens on doit utiliser une **méthode associée aux vectoriseurs** de la librairie ([notamment `CountVectorizer`](https://scikit-learn.org/stable/modules/feature_extraction.html#common-vectorizer-usage)). Plus précisément, les vectoriseurs de cette libraire utilisent la méthode `build_analyzer` ([qui appelle implicitement la méthode `build_tokenizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html#sklearn.feature_extraction.text.CountVectorizer.build_analyzer)). *C'est uniquemement dans un objectif d'illustration*, qu'on rend l'opération de tokénisation explicite dans cette sous-section : en pratique on utilisera plutôt une autre méthode du vectoriseur *après la vectorisation* (cf. plus bas).


Par défaut, tous les mots de moins 2 lettres sont supprimés et toute ponctuation ou espace est considéré comme un séparateur de token, donc est également supprimé. De plus, les caractères sont transformés en minuscules :


In [48]:
#On importe le vectoriseur qu'on place dans la variable vectorizer

from sklearn.feature_extraction.text import CountVectorizer

vectorizer=CountVectorizer()

#Le tokéniseur est récupéré par la méthode .build_analyzer()

tokenizer_sklearn=vectorizer.build_analyzer() #.build_tokenizer() peut s'appliquer, sans accepter toutes les options du vectorizer

#On l'applique alors au texte

result1 = tokenizer_sklearn(french_sent)

print(result1)

['le', 'traitement', 'automatique', 'du', 'langage', 'naturel', 'abr', 'taln', 'ou', 'traitement', 'automatique', 'de', 'la', 'langue', 'naturelle', 'ou', 'encore', 'traitement', 'automatique', 'des', 'langues', 'abr', 'tal', 'est', 'un', 'domaine', 'multidisciplinaire', 'impliquant', 'la', 'linguistique', 'informatique', 'et', 'intelligence', 'artificielle', 'qui', 'vise', 'créer', 'des', 'outils', 'de', 'traitement', 'de', 'la', 'langue', 'naturelle', 'pour', 'diverses', 'applications']


<div class="alert alert-info">
<b> Info :</b> On peut spécifier des options lors de la tokénisation/vectorisation. Par exemple, on désactive la mise en minuscule du texte avec l'option <code>lowercase=False</code> au moment de la déclaration de la variable <code>vectorizer</code>. On verra dans le chapiter 2 comment spécifier des <code>regex</code> pour la tokénisation. On peut même désactiver ce tokéniseur pour en utiliser un autre (cf. plus bas).</div>


- **La segmentation des phrases** n'est pas explicitement prévu dans cette librairie qui vectorisé les documents comme une seule phrase (dans une logique *Bags of Words*, cf. plus bas).

<div class="alert alert-success">
<b> Done !</b> Vous savez désormais découper un texte en tokens, sachant que cette unité d'analyse a un impact majeur sur la performance du modèle...
</div>

# 2. Normalisation

La phase dite de normalisation n'est pas obligatoire, mais est souvent conseillée **pour limiter la variété des tokens retenus pour l'analyse** (du fait de la ["Malédiction de la dimension"](https://fr.wikipedia.org/wiki/Fl%C3%A9au_de_la_dimension) : faire de l'inférence depuis un nombre réduit d'expériences dans un espace de possibilités de dimension élevée). De plus, les matrices termes-documents issues de (beaucoup de) textes sont composées d'un grand nombre de 0 (on parle de matrices creuses ou éparses, "sparse matrix" en anglais), ce qui peut fortement détériorer la précision des estimations.

On utilise uniquement `SpaCy` ici car les étapes les plus fréquentes de normalisation sont possibles dans beaucoup de langues et très faciles à réaliser.

<div class="alert alert-info">
<b> Info :</b> Si aucune des étapes suivantes n'est utile, on peut considérer (selon le cas d'usage) de mettre tous les caractères en minuscules (pour que "Mot" et "mot" ne soient pas considérés comme deux entrées différentes dans la matrice des termes des documents). La méthode <code>lower</code> associée aux chaînes de caractères dans Python permet de faire cela : <code>[token.text.lower() for token in doc_spacy2]</code>.
</div>

In [55]:
print([token.text.lower() for token in doc_spacy2])

['le', 'taln', 'est', 'sorti', 'des', 'laboratoires', 'de', 'recherche', 'pour', 'être', 'progressivement', 'mis', 'en', 'œuvre', 'dans', 'des', 'applications', 'informatiques', 'nécessitant', "l'", 'intégration', 'du', 'langage', 'humain', 'à', 'la', 'machine', '.', 'aussi', 'le', 'taln', 'est', '-il', 'parfois', 'appelé', 'ingénierie', 'linguistique', '.', 'en', 'france', ',', 'le', 'traitement', 'automatique', 'de', 'la', 'langue', 'naturelle', 'a', 'sa', 'revue', ',', 'traitement', 'automatique', 'des', 'langues', ',', 'publiée', 'par', 'l’', 'association', 'pour', 'le', 'traitement', 'automatique', 'des', 'langues', '(', 'atala', ')', '.']


## Stopwords et ponctuation

- Les stopwords (ou "mots vides" en français) correspondent aux *mots particulièrement fréquents dans une langue (ou un cas d'usage)* et qui ont peu de sens en eux mêmes : ils n'aident pas à comprendre le contenu d'un document. Leur suppression  avec `SpaCy` est assez immédiate, mais il peut être préférable de procéder de façon manuelle ou ne pas les retirer (selon le cas d'usage). 

La méthode spécifique aux stopwords est `is_stop` :

In [58]:
for token in doc_spacy2[:10]:
    print("'{}' est un stopword : {}".format(token.text,token.is_stop))

'Le' est un stopword : True
'TALN' est un stopword : False
'est' est un stopword : True
'sorti' est un stopword : False
'des' est un stopword : True
'laboratoires' est un stopword : False
'de' est un stopword : True
'recherche' est un stopword : False
'pour' est un stopword : True
'être' est un stopword : True


Après filtrage de ces termes, on s'aperçoit qu'une certaine interprétation du contenu est toujours possible :

In [60]:
#Equivalent à créer un boucle avec une condition :
#test2=[]
#for elem in doc_spacy2:
#    if not elem.is_stop:
#        test2.append(elem.text)
#print(test2)

In [61]:
#Suppression des stopwords
print([token.text for token in doc_spacy2 if not token.is_stop])

['TALN', 'sorti', 'laboratoires', 'recherche', 'progressivement', 'mis', 'œuvre', 'applications', 'informatiques', 'nécessitant', 'intégration', 'langage', 'humain', 'machine', '.', 'TALN', '-il', 'appelé', 'ingénierie', 'linguistique', '.', 'France', ',', 'traitement', 'automatique', 'langue', 'naturelle', 'revue', ',', 'Traitement', 'automatique', 'langues', ',', 'publiée', 'Association', 'traitement', 'automatique', 'langues', '(', 'ATALA', ')', '.']


- Pour la ponctuation, une méthode équivalente existe : `is_punct`. On peut cumuler les deux avec une condition additionnelle :

In [63]:
#Suppression de la ponctuation en plus des stopwords :
print([token.text for token in doc_spacy2 if not token.is_stop and not token.is_punct])

['TALN', 'sorti', 'laboratoires', 'recherche', 'progressivement', 'mis', 'œuvre', 'applications', 'informatiques', 'nécessitant', 'intégration', 'langage', 'humain', 'machine', 'TALN', '-il', 'appelé', 'ingénierie', 'linguistique', 'France', 'traitement', 'automatique', 'langue', 'naturelle', 'revue', 'Traitement', 'automatique', 'langues', 'publiée', 'Association', 'traitement', 'automatique', 'langues', 'ATALA']


<div class="alert alert-info">
<b> Info :</b> <code>NLTK</code> fournit des listes de stopwords dans plusieurs langues : <a href=https://stackoverflow.com/a/5486535/12910854>on peut en utiliser une comme condition dans la création des listes de tokens</a>.  <code>NLTK</code> ne fournit cependant pas de liste de ponctuation : <a href=https://stackoverflow.com/a/45516258/12910854>on peut importer la librairie <code>string</code> qui contient la liste <code>punctuation</code> pour construire une condition additionnelle</a>.
</div>

## Lemmatisation

La lemmatisation correspond au processus de *remplacement d'un mot par la forme qui l'a généré* (son "lemme"). L'objectif est la réduction de la variété des tokens trouvés dans le corpus en les regroupant en un seul (ex : "suis" et "est" sont lemmatisés vers "être"). 

De nouveau, c'est assez immédiat avec de `SpaCy`, par la méthode `.lemma_`, même s'il est parfois préférable de ne pas lemmatiser un texte (selon le cas d'usage) :

In [67]:
for token in doc_spacy2[:10]:
    print("'{}' a pour lemme : {}".format(token.text,token.lemma_))

'Le' a pour lemme : le
'TALN' a pour lemme : taln
'est' a pour lemme : être
'sorti' a pour lemme : sortir
'des' a pour lemme : un
'laboratoires' a pour lemme : laboratoire
'de' a pour lemme : de
'recherche' a pour lemme : recherche
'pour' a pour lemme : pour
'être' a pour lemme : être


On peut voir que cette méthode transforme les caractères en lettre minuscule (sauf ceux qui sont considérés comme des noms propres) :

In [69]:
#Cumul des normalisations
print([token.lemma_ for token in doc_spacy2 if not token.is_stop and not token.is_punct])

['taln', 'sortir', 'laboratoire', 'recherche', 'progressivement', 'mettre', 'œuvre', 'application', 'informatique', 'nécessiter', 'intégration', 'langage', 'humain', 'machine', 'taln', 'il', 'appeler', 'ingénierie', 'linguistique', 'France', 'traitement', 'automatique', 'langue', 'naturel', 'revue', 'traitement', 'automatique', 'langue', 'publier', 'association', 'traitement', 'automatique', 'langue', 'ATALA']


<div class="alert alert-info">
<b> Info:</b>  <code>NLTK</code> ne fournit pas de lemmatiseur pour le français. Pour l'anglais, il y a <code>WordNetLemmatizer</code> qui peut être instancié sur une chaîne de caractères avec la méthode <code>lemmatize</code>.
</div>

RQ : en cumulant les trois étapes de normalisation précédentes on réduit de près de la moitié la taille du vocabulaire :

In [72]:
taille_init = len(set(token.text for token in doc_spacy2))
taille_reduite = len(set(token.lemma_ for token in doc_spacy2 if not token.is_stop and not token.is_punct))
(taille_init-taille_reduite)/taille_init

0.49056603773584906

## Stemming

La stemmisation (ou encore racinisation) est une opération légèrement différente de la lemmatisation : *on coupe les mots pour ne garder que leur racine*, notamment en retirant les préfixes et suffixes. Cette opération est donc plus "radicale" que la lemmatisation en termes de modification du texte et modifie leurs sens pour l'être humain. 

Cependant, à l'étape de la modélisation, **ça peut être une bonne solution à la place de la lemmatisation** selon Bengfort et al. (2018, p. 72) : le stemming a l'avantage d'être a priori plus rapide, car les algorithmes de lemmatisation réalisent le plus souvent des boucles au sein d'un dictionnaire ou une base de données et exploitent également les fonctions grammaticales des mots dans le texte.

La librairie `SpaCy` ne propose pas de méthode de stemming [car la lemmatisation est globalement préférée par ses développeurs](https://github.com/explosion/spaCy/issues/327#issuecomment-208164148). Si on souhaite faire du stemming, on peut cependant utiliser l'algorithme "de Porter" (prévu uniquement pour l'anglais) ou Snowball (développé par le même auteur mais conçu pour plusieurs langues) ([voir cette page pour plus de détails](https://www.nltk.org/api/nltk.stem.html)). Les deux sont implémentés dans `NLTK`, et facilement intégrables aux étapes de normalisation vues ci-dessus :

In [76]:
from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer(language='french')

In [77]:
print([stemmer.stem(token.text) for token in doc_spacy2 if not token.is_stop and not token.is_punct])

['taln', 'sort', 'laboratoir', 'recherch', 'progress', 'mis', 'œuvr', 'appliqu', 'informat', 'nécessit', 'integr', 'langag', 'humain', 'machin', 'taln', '-il', 'appel', 'ingénier', 'linguist', 'franc', 'trait', 'automat', 'langu', 'naturel', 'revu', 'trait', 'automat', 'langu', 'publi', 'associ', 'trait', 'automat', 'langu', 'atal']


<div class="alert alert-success">
<b> Done !</b> La normalisation est faite.
</div>

# 3. Vectorisation des textes

Cette étape consiste en la préparation du texte normalisé (ou pas) : **les listes de tokens sont transformés en des vecteurs de nombres en vue d'une analyse par un ou plusieurs algorithmes**. 

Parmi un grand nombre de méthodes possibles, on se limite dans ce notebook aux *encodages one-hot*, *de fréquence* et *TF-IDF* qui sont les plus simples à appréhender et souvent les premières utilisées en NLP. Ces trois méthodes sont des approches dites par "sacs de mots" ("**Bags of Words**" en anglais) telles qu'on fait l'hypothèse que les tokens sont distribués de façon indépendante : 

> la présence d'un token, noté $B$, n'influence pas la probabilité d'apparition dans le document d'un autre, noté $A$, c'est-à-dire que $P(A|B) = P(A)$. 

On reviendra sur cette hypothèse dans le notebook suivant, mais elle implique que des documents contenant les mêmes tokens dans des sens différents sont vectorisés de façon identique.

## Encodages "one-hot" et de fréquence

L'encodage **one-hot** crée des vecteurs de 0 et 1 pour chaque document dans des vecteurs dont la dimension est déterminée par le vocabulaire du corpus : un 0 est associé à l'absence d'un token dans un document qui est présent dans le corpus et un 1 indique sa présence également dans le document. 

La figure suivante illustre cette méthode en supposant 3 documents vectorisés en anglais (chacun étant composé d'une seule phrase).

<figure>
    <div align="center" style="margin-top:0.5cm"> <b> Vectorisation par la méthode d'encodage binaire avec lemmatisation, retrait de la ponctuation mais pas des stopwords (Bengfort et al., 2018, p.60)</b> </div>
  <img src="img/vect_one-hot.png" style="margin-top:0.5cm;margin-bottom:0cm;width:80%">
</figure>

### Version intégrée dans Scikit-Learn

Parmi les objets de `Scikit-Learn` qui vectorisent les textes, il y a `CountVectorizer` (qu'on a présenté brièvement plus haut). C'est ce qu'on utilise pour l'encodage "one-hot" en intégrant l'option `binary=True` au moment de la déclaration de la variable `vectorizer`. De plus, pour effectivement générer les vecteurs, il faut appliquer aux textes les méthodes `fit` et `transform` ou alors la méthode `fit_transform` qui les enchaîne automatiquement.

<div class="alert alert-danger">
<b> Attention :</b> Il faut donner à <code>CountVectorizer</code> un liste de textes en entrée et non du texte brut. On peut donc encadrer un texte brut par des crochets <code>[...]</code>, mais cela génère un seul vecteur (sans intérêt). </div>


On considère ici une liste de textes appelée `corpus` et composée des phrases du texte brut défini plus haut :

In [88]:
corpus = [doc.text for doc in doc_spacy2.sents]
corpus

["Le TALN est sorti des laboratoires de recherche pour être progressivement mis en œuvre dans des applications informatiques nécessitant l'intégration du langage humain à la machine.",
 'Aussi le TALN est-il parfois appelé ingénierie linguistique.',
 'En France, le traitement automatique de la langue naturelle a sa revue, Traitement automatique des langues, publiée par l’Association pour le traitement automatique des langues (ATALA).']

Une fois l'algorithme appliqué aux documents, on peut récupérer facilement la liste des tokens avec la méthode `.get_feature_names_out` :

In [90]:
from sklearn.feature_extraction.text import CountVectorizer
 
#L'option binary implique la création de 0 et 1 dans les vecteurs créés 
vectorizer=CountVectorizer(binary=True)

#fit_transform est une méthode commune à plusieurs objets Scikit-Learn qui instancie les tâches associées à l'objet (ici CountVectorizer) 

vectors=vectorizer.fit_transform(corpus)

print("Le corpus est constitué de {} documents et a un vocabulaire de {} :\n".format(vectors.shape[0],vectors.shape[1]))

#Le vectoriseur "fitté" (adapté) contient le vocabulaire du corpus
print(vectorizer.get_feature_names_out())

print("\n Les vecteurs bruts sont visualisés avec la méthode toarray :\n")
print(vectors.toarray())

Le corpus est constitué de 3 documents et a un vocabulaire de 42 :

['appelé' 'applications' 'association' 'atala' 'aussi' 'automatique'
 'dans' 'de' 'des' 'du' 'en' 'est' 'france' 'humain' 'il' 'informatiques'
 'ingénierie' 'intégration' 'la' 'laboratoires' 'langage' 'langue'
 'langues' 'le' 'linguistique' 'machine' 'mis' 'naturelle' 'nécessitant'
 'par' 'parfois' 'pour' 'progressivement' 'publiée' 'recherche' 'revue'
 'sa' 'sorti' 'taln' 'traitement' 'être' 'œuvre']

 Les vecteurs bruts sont visualisés avec la méthode toarray :

[[0 1 0 0 0 0 1 1 1 1 1 1 0 1 0 1 0 1 1 1 1 0 0 1 0 1 1 0 1 0 0 1 1 0 1 0
  0 1 1 0 1 1]
 [1 0 0 0 1 0 0 0 0 0 0 1 0 0 1 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0 0 0 0
  0 0 1 0 0 0]
 [0 0 1 1 0 1 0 1 1 0 1 0 1 0 0 0 0 0 1 0 0 1 1 1 0 0 0 1 0 1 0 1 0 1 0 1
  1 0 0 1 0 0]]


<div class="alert alert-info">
<b> Info:</b> Si on enlève l'option <code>binary=True</code>, on obtient un autre encodage, dit <strong>de fréqeunce</strong> ou <strong>termes-fréquences</strong> : les 1 de l'encodage one-hot sont simplement remplacés par le nombre d'apparitions du token dans le document. Ce second encodage est assez peu utilisé par rapport au premier et à l'encodage TF-IDF (étudié plus bas).
</div>

### Scikit-Learn avec un tokéniseur tiers

Comme nous l'avons vu plus haut, la librairie `Scikit-Learn` réalise des opérations assez limitées sur les chaînes de caractères et on peut souhaiter utiliser un tokéniseur tiers comme celui de `SpaCy`.

Pour cela on désactive le tokéniseur de `Scikit-Learn` avec une astuce astuce assez simple : on lui spécifie une fonction "identité" (appelée ici `identity`) qui retourne simplement l'objet soumis, sans le modifier. Il suffit alors que l'objet en question soit la liste de listes de tokens créée par `Spacy`.

In [94]:
#Une fonction que ne fait rien que retourné l'objet entré
def identity(sentence):
    return sentence

In [95]:
#On crée un corpus des 3 documents d'une phrase chacun, auxquels on applique les méthodes de SpaCy vues plus haut
corpus_spacy=[[word.lemma_ for word in doc if not word.is_stop and not word.is_punct] for doc in doc_spacy2.sents]
print(corpus_spacy)

[['taln', 'sortir', 'laboratoire', 'recherche', 'progressivement', 'mettre', 'œuvre', 'application', 'informatique', 'nécessiter', 'intégration', 'langage', 'humain', 'machine'], ['taln', 'il', 'appeler', 'ingénierie', 'linguistique'], ['France', 'traitement', 'automatique', 'langue', 'naturel', 'revue', 'traitement', 'automatique', 'langue', 'publier', 'association', 'traitement', 'automatique', 'langue', 'ATALA']]


In [96]:
#Vectoriseur accompagné de lowercase=False (à modifier au niveau de SpaCy maintenir des majuscules) et avec tokeniseur substitué

vectorizer2=CountVectorizer(binary=True, lowercase=False, tokenizer=identity)

vectors2=vectorizer2.fit_transform(corpus_spacy)

print("Le corpus est constitué de {} documents et a un vocabulaire de {} :\n".format(vectors2.shape[0],vectors2.shape[1]))

print(vectorizer2.get_feature_names_out())

print("\n Les vecteurs bruts sont visualisés avec la méthode toarray :\n")
print(vectors2.toarray())

print("\nRQ : le vocabulaire est réduit de {:.2f} % avec la normalisation de SpaCy !".format(100*(vectors.shape[1]-vectors2.shape[1])/vectors.shape[1]))

Le corpus est constitué de 3 documents et a un vocabulaire de 27 :

['ATALA' 'France' 'appeler' 'application' 'association' 'automatique'
 'humain' 'il' 'informatique' 'ingénierie' 'intégration' 'laboratoire'
 'langage' 'langue' 'linguistique' 'machine' 'mettre' 'naturel'
 'nécessiter' 'progressivement' 'publier' 'recherche' 'revue' 'sortir'
 'taln' 'traitement' 'œuvre']

 Les vecteurs bruts sont visualisés avec la méthode toarray :

[[0 0 0 1 0 0 1 0 1 0 1 1 1 0 0 1 1 0 1 1 0 1 0 1 1 0 1]
 [0 0 1 0 0 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0]
 [1 1 0 0 1 1 0 0 0 0 0 0 0 1 0 0 0 1 0 0 1 0 1 0 0 1 0]]

RQ : le vocabulaire est réduit de 35.71 % avec la normalisation de SpaCy !


C:\Users\34550\.conda\envs\txtm_nlp2\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


## Encodage TF-IDF

L'encodage **TF-IDF** diffère de la vectorisation one-hot en ce qu'elle utilise, au lieu des 1, des nombres réels qui reflètent la spécificité de la présence d'un token dans un document (par rapport aux autres documents). Plus précisément, cette valeur est calculée pour un token donné dans un document comme le produit de la **term frequency** (TF) par la **inverted document frequency** (IDF). 

TF est égale au nombre de fois qu'un token donné apparaît dans un document et IDF est donnée par la formule :

$$
IDF(\text{token}) = log\left(\frac{m}{DF(\text{token})}\right),
$$

où $m$ est le nombre total de documents dans le corpus et $DF(\text{token})$ est le nombre de documents qui contiennent le token au moins une fois. Ainsi :


$$
\text{$TF$-$IDF$(token)} = TF\text{(token)} \times [log(m) - log(DF\text{(token)})].
$$

Plus le TF-IDF d'un token est important, plus il est "caractéristique" dans le document relativement à ce qu'il l'est dans les autres documents. En effet ce nombre augmente avec la fréquence du mot dans le document (via TF) mais diminue avec la fréquence du mot dans le corpus (via DF).

La figure ci-dessous illustre cette méthode avec 3 documents en anglais.
<figure>
    <div align="center" style="margin-top:0.5cm"> <b> Vectorisation par la méthode d'encodage TF-IDF avec lemmatisation, retrait de la ponctuation mais pas des stopwords (Bengfort et al., 2018, p.62)</b> <br>
  <img src="img/vect_tfidf.png" style="margin-top:0.5cm;margin-bottom:0cm;width:80%"></div>
</figure>

L'implémentation de la méthode TF-IDF dans `Scikit-Learn` est immédiate, même si quelques modifications pratiques sont apportées à la formule ci-dessus ainsi qu'une normalisation ([voir la documentation pour davantage d'information](https://scikit-learn.org/stable/modules/feature_extraction.html#tfidf-term-weighting)). 

In [101]:
from sklearn.feature_extraction.text import TfidfVectorizer

#De nouveau, on peut intégrer les résultats de la librairie SpaCy

vectorizer3=TfidfVectorizer(lowercase=False, tokenizer=identity)

vectors3=vectorizer3.fit_transform(corpus_spacy)

print("Le corpus est constitué de {} documents  et a un vocabulaire de {} :\n".format(vectors3.shape[0],vectors3.shape[1]))

print(vectorizer3.get_feature_names_out())

print("\n Les vecteurs bruts sont visualisés avec la méthode toarray :\n")
print(vectors3.toarray())

Le corpus est constitué de 3 documents  et a un vocabulaire de 27 :

['ATALA' 'France' 'appeler' 'application' 'association' 'automatique'
 'humain' 'il' 'informatique' 'ingénierie' 'intégration' 'laboratoire'
 'langage' 'langue' 'linguistique' 'machine' 'mettre' 'naturel'
 'nécessiter' 'progressivement' 'publier' 'recherche' 'revue' 'sortir'
 'taln' 'traitement' 'œuvre']

 Les vecteurs bruts sont visualisés avec la méthode toarray :

[[0.         0.         0.         0.27137867 0.         0.
  0.27137867 0.         0.27137867 0.         0.27137867 0.27137867
  0.27137867 0.         0.         0.27137867 0.27137867 0.
  0.27137867 0.27137867 0.         0.27137867 0.         0.27137867
  0.20639047 0.         0.27137867]
 [0.         0.         0.46735098 0.         0.         0.
  0.         0.46735098 0.         0.46735098 0.         0.
  0.         0.         0.46735098 0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.35543247 0.         0.   

## Stockage des vecteurs dans un Dataframe

Si on veut stocker les vecteurs dans un Dataframe de la librairie `Pandas`, on procède de la façon suivante :

In [104]:
import pandas as pd
df=pd.DataFrame(vectors2.toarray(),index=None,columns=vectorizer2.get_feature_names_out())
df

,ATALA,France,appeler,application,association,automatique,humain,il,informatique,ingénierie,...,naturel,nécessiter,progressivement,publier,recherche,revue,sortir,taln,traitement,œuvre
0,0,0,0,1,0,0,1,0,1,0,...,0,1,1,0,1,0,1,1,0,1
1,0,0,1,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,1,0,0
2,1,1,0,0,1,1,0,0,0,0,...,1,0,0,1,0,1,0,0,1,0


<div class="alert alert-info">
<b> Info:</b>  Si on a des noms de documents spécifiques dans une liste (ex : <code>names=['doc1', 'doc2', 'doc3']</code>), on peut l'intégrer en tant qu'index du Dataframe en spécifiant  <code>index=names</code>. On peut alors extraire un document à partir de son nom (ex: <code>df.loc[['doc2']]</code>).
</div>

<div class="alert alert-success">
<b> Done !</b> Vous savez désormais transformer des textes en vecteurs issus de textes et pouvez y appliquer des algorithmes !
</div>

# Reconnaissance des entités nommées (NER)

Les entités nommées (*Named entity* ou NE en anglais) sont des références à des éléments du monde réel (noms propres, lieux, dates, etc.). Pour leur détection (***Named Entity Recognition*** ou NER en anglais), il peut être nécessaire de construire des règles de type Regex (majuscules, etc.) et/ou d'entrainer un algorithme spécifique.

Ici, on se limite à l'application de l'algorithme préentrainé de `SpaCy`. Ce package exploite implicitement des fonctions d'étiquetage lexicale (en anglais Part-Of-Speech tagging, [POS tagging](https://en.wikipedia.org/wiki/Part-of-speech_tagging)) du français, [toutes listées dans l'onglet "Label Scheme" des modèles](https://spacy.io/models/fr#fr_core_news_lg), ainsi que [l'analyse des dépendances des termes dans le pipeline de son modèle](https://spacy.io/usage/linguistic-features#dependency-parse). La présentation des relations sous forme de graph peut être faite avec le module `displacy` qui est un package du ["SpaCy Universe"](https://spacy.io/universe/project/displacy) et détaillé [sur le site de `SpaCy`](https://spacy.io/usage/visualizers#dep). On ne la voit pas ici.

## Utilisation de SpaCy

Ce package détecte automatiquement les NE et associe également à chacune un label : `LOC` pour les lieux, `ORG` pour organisation, `PER` pour personne et `MISC` pour les entités diverses ("miscellaneous"). 

On récupère les NE avec la méthode `ents` de chaque objet `Doc`, et leur label avec la méthode `label_` de chaque NE :

In [111]:
#Considérons les avis suivants sur des films récupérés de Allocine.fr :
reviews=['''Ce premier volet de la mythique saga est déjà un film très bon et très sympa à voir, avec un Sean Connery 
            parfait et une Ursula Andress divine. Relaxant.''',
         "Film trop lent comme ses dialogues. Je n'ai pas du tout accroché.",
         '''Premier film de la saga Kozure Okami, "Le Sabre de la vengeance" est un très bon film qui mêle drame et action, 
            et qui, en 40 ans, n'a pas pris une ride.'''
        ]

In [112]:
#Une fois le modèle appliqué, tout token est catégorisé comme étant ou non une entité, et si oui de quel type

for n, review in enumerate(reviews):
    
    doc=nlp(review)
    
    entities=doc.ents
    
    print("Dans le document {}, SpaCy a trouvé {} entités :\n".format(n+1, len(entities)))
    
    for ent in entities:
        
        print('-', ent.text, '->', ent.label_,'\n')
    

Dans le document 1, SpaCy a trouvé 3 entités :

- Sean Connery 
             -> PER 

- Ursula Andress -> PER 

- Relaxant -> PER 

Dans le document 2, SpaCy a trouvé 0 entités :

Dans le document 3, SpaCy a trouvé 2 entités :

- Kozure Okami -> MISC 

- Le Sabre de la vengeance -> MISC 



<div class="alert alert-info">
<b> Info:</b> <code>SpaCy</code> indique des scores assez hauts (<a href=https://spacy.io/models/fr#fr_core_news_lg>dans l'onglet "Acurracy Evaluation" du modèle</a>) : Precision = 0.84, Recall = 0.84 and F-score = 0.84.  On peut adapter (update) l'algorithme de <code>SpaCy</code> avec des exemples additionnels, <a href=https://spacy.io/usage/training#config-custom>comme pour toute autre étape des modèles de ce package</a>, pour améliorer ces scores à son cas d'usage ou pour détecter de nouveaux types de NE.
</div>

## Visualisation des NE

Si on le souhaite, `Displacy` peut être utilisé pour afficher le résultat de cette tâche en utilisant l'option `style` :

In [116]:
from spacy import displacy
displacy.render(nlp(reviews[0].replace('\n','')), style="ent")

<div class="alert alert-success">
<b> Done !</b> Vous savez comment réaliser la détection les entités nommées. Des modèles de transformers seront vus dans le notebook suivant.
</div>

# Ham or Spam ? That is the Python Question !

Cette section illustre comment **anlayser les vecteurs issus de documents textuels par un algorithme** très simple : la régression logistique ([voir la page dédiée de `Scikit-Learn` pour plus d'information](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)).

Pour cela, on considère un cas d'usage classique de détection de spams dans un ensemble d'emails, en réalisant :
- l'import d'une base de données tabulaires (fichier `.csv`) contenant une variable de texte et une autre avec son label (vraie valeure : spam ou pas),
- découpage de la base de données en deux échatillons d'entrainement et de test (pas de validation car pas de recherche d'hyperparamètres),
- l'entrainement de l'algorithme sur des textes tokénisé de façon automatique par `Scikit-Learn`, et
- la mesure de la performance de l'algorithme entrainé.

Il s'agit bien sûr d'une simple illustration de classification de documents, et les méthodes plus avancées seront vues dans le prochain notebook.

## Import d'une base d'exemples et échantillonnage

### Import d'une base d'exemples

On importe avec `pandas` le fichier `spam_ham_dataset.csv` trouvé sur le [site Kaggle](https://www.kaggle.com/datasets/venky73/spam-mails-dataset) qui contient des exemples de spams et non-spams (*hams*) :

In [123]:
import pandas as pd
df=pd.read_csv('data/spam_ham_dataset.csv', sep=',')
df

,id,label,text,label_num
0,605,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
1,2349,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,3624,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,4685,spam,"Subject: photoshop , windows , office . cheap ...",1
4,2030,ham,Subject: re : indian springs\r\nthis deal is t...,0
...,...,...,...,...
5166,1518,ham,Subject: put the 10 on the ft\r\nthe transport...,0
5167,404,ham,Subject: 3 / 4 / 2000 and following noms\r\nhp...,0
5168,2933,ham,Subject: calpine daily gas nomination\r\n>\r\n...,0
5169,1409,ham,Subject: industrial worksheets for august 2000...,0


On crée alors des vecteurs au sens de `numpy` contenant les labels en variable à expliquer et les textes en variable explicative :

In [125]:
import numpy as np
X = np.array(df['text'].to_list())
y = np.array(df['label'].to_list()) #on peut ici utiliser indistinctement df['label_num  ou df['label_num']']  ou df['label']

On peut alors accéder au contenu d'un mail et à son label leur index :

In [127]:
n=2
print(y[n])
print(X[n])

ham
Subject: neon retreat
ho ho ho , we ' re around to that most wonderful time of the year - - - neon leaders retreat time !
i know that this time of year is extremely hectic , and that it ' s tough to think about anything past the holidays , but life does go on past the week of december 25 through january 1 , and that ' s what i ' d like you to think about for a minute .
on the calender that i handed out at the beginning of the fall semester , the retreat was scheduled for the weekend of january 5 - 6 . but because of a youth ministers conference that brad and dustin are connected with that week , we ' re going to change the date to the following weekend , january 12 - 13 . now comes the part you need to think about .
i think we all agree that it ' s important for us to get together and have some time to recharge our batteries before we get to far into the spring semester , but it can be a lot of trouble and difficult for us to get away without kids , etc . so , brad came up with a p

### Création des échantillons train et test

On partitionne alors la base données en observations qui seront soit utilisées pour l'entrainement de l'algorithme (et le choix des hyperparamètres) soit pour en mesurer la performance.

Cette opération peut être faite à la main mais c'est plus pratique avec [la méthode `train_test_split` de `Scikit-Learn`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) : on peut facilement s'assurer d'avoir des proportions identiques des catégories dans les deux échantillons (option `stratify`) et de permettre la reproduction du découpage aléatoire (option `random_state` qui fixe la graine d'aléa) :

In [130]:
from sklearn.model_selection import train_test_split

#fixation de la graine d'aléa (qui n'est pas un hyperparamètre)
RANDOM = 42

# On garde 70 % des observations pour l'entrainement

TEST = 0.30

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST, stratify=y, random_state=RANDOM)

print("L'algorithme est entrainé à partir de {} exemples".format(len(X_train)))
print("L'évaluation de l'algorithme peut être faite sur {} exemples".format(len(X_test)))

L'algorithme est entrainé à partir de 3619 exemples
L'évaluation de l'algorithme peut être faite sur 1552 exemples


## Entrainement du classifieur de spams

On entraine une régression logistique en guise d'algorithme basique à l'intérieure, [dans un objet `Pipeline`](https://scikit-learn.org/stable/modules/compose.html), après l'étape de vectorisation one-hot de `Scikit-Learn` (avec le tokéniseur et la normalisation par défaut) :

In [133]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

#L'objet pipeline de Sklearn permet d'appliquer les différentes étapes de la vectorisation à l'entrainement en cascade

#https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

vector_clf = Pipeline([
    ('vector', CountVectorizer(binary=True)),
    ('clf', LogisticRegression(n_jobs=-1,  #pour utiliser tous les processeurs
                              verbose=1)), #degré d'information durant l'entrainement
])

#La pipeline est instanciée sur le train set avec la méthode fit() (car LogisticRegression ne transforme pas les vecteurs)

vector_clf.fit(X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.


Pipeline(steps=[('vector', CountVectorizer(binary=True)),
                ('clf', LogisticRegression(n_jobs=-1, verbose=1))])

## Evaluation de performance

Le modèle est désormais entrainé et on peut par exemple l'appliquer à un texte de l'ensemble de validation pour vérifier le résultat :

In [136]:
choix_num = 1551 #entre 0 et 1551 pour 1552 observations de validation

texte = X_test[choix_num]
print("Contenu du message :\n\n",texte,'\n')

label = y_test[choix_num]
print("Label réel : {} \n".format(label))

prediction = vector_clf.predict([texte, ])
print("Label prédit : {} \n".format(prediction[0]))

if label == prediction:
    print('Bonne prédiction!') 
else:
    print('Mauvaise prédiction :(')

Contenu du message :

 Subject: ordering this pain medication
page is loading . .
image not showing ? see message here .
as seen on cbs news , fox news and oprah ,
your cheapest source for viagra , cialis , xanax ,
valium and hundreds more top - quality medication !
look here
expunge my address 2
% rnd _ body 

Label réel : spam 

Label prédit : spam 

Bonne prédiction!


Pour apprécier la performance d'un classifieur, il est cependant nécessaire de considérer des scores appliqués à l'ensemble de l'échatillon de test, et non seulement à quelques exemples. Là encore Scikit-Learn permet facilement de calculer les principaux scores avec le module `metrics` :

In [138]:
from sklearn import metrics

# Prédiction sur le train-set
y_pred = vector_clf.predict(X_train)
acc_train = metrics.accuracy_score(y_train, y_pred)
print("Accuracy sur le train-set : {:.5f}".format(acc_train))

# Prédiction sur le validation-set
y_pred = vector_clf.predict(X_test)
acc_test = metrics.accuracy_score(y_test, y_pred)
print("\nAccuracy sur le test-set: {:.5f}".format(acc_test))

# Indication sur le sur-apprentissage
print("\nPourcentage de variation d'accuracy entre le train et test: {:.2f}".format((acc_test-acc_train)/acc_train*100))

print("\nRapport de classification sur le test-set:\n\n",metrics.classification_report(y_test, y_pred))

Accuracy sur le train-set : 0.99945

Accuracy sur le test-set: 0.98003

Pourcentage de variation d'accuracy entre le train et test: -1.94

Rapport de classification sur le test-set:

               precision    recall  f1-score   support

         ham       0.99      0.99      0.99      1102
        spam       0.96      0.97      0.97       450

    accuracy                           0.98      1552
   macro avg       0.98      0.98      0.98      1552
weighted avg       0.98      0.98      0.98      1552



<div class="alert alert-info">
<b> Info:</b> La figure suivante explique trois scores pour la classification dans une catégorie (<a href=https://fr.wikipedia.org/wiki/Pr%C3%A9cision_et_rappel>inspirée de Wikipédia</a>).
<div align="center" style="margin-top:0.5cm">
<img src="img/scores_classif.png" style="margin-top:0.5cm;margin-bottom:0.5cm;width:70%">
</div>
Accuracy (en français justesse ou exactitude) : part des textes classés correctement
$$
Acc = \frac{T_P + T_N}{T_P + T_N + F_P + F_N}
$$
    
Precision : part des prédictions dans une classe visée qui sont correctes 
$$
Prec = \frac{T_P}{T_P + F_P}
$$

Recall : part des vraies valeurs d'une classe visée qui sont récupérées 
$$
Rec = \frac{T_P}{T_P + F_N}
$$
    
F-Score : moyenne harmonique entre Precision et Recall
$$
F_1 = 2 \frac{Prec * Rec}{Prec + Rec}
$$
    
Les résultats "macro" et "pondérés" sont des moyennes des scores calculés par catégorie (<a href=https://scikit-learn.org/stable/modules/model_evaluation.html>voir la documentation pour le détail des méthodes exactes</a>).

<br>
    
<b>Remarque :</b> dans certains cas, il peut être préférable de baser son choix du modèle par rapport aux probabilités qui déterminent les classes prédites plutôt que les classes elles-même. On ne présente pas cette approche ici.
</div>

Pour information, on peut ensuite récupérer les coeffeicients de la régression avec la méthode `coef_` de l'étape `clf` du pipeline, et appliquer la méthode `get_feature_names_out` depuis l'étape `vector`. La méthode `zip` de python permet d'associer les deux listes obtenues.

In [141]:
coef=vector_clf['clf'].coef_.tolist()[0]
vocab=vector_clf['vector'].get_feature_names_out().tolist()
list(zip(coef,vocab))[5000:]

[(0.0004151057387027048, 'alimony'),
 (0.011267397360869392, 'alink'),
 (0.00665034386754708, 'aliow'),
 (0.02080474939394673, 'aliphatic'),
 (0.04309524502582661, 'aliquot'),
 (0.029885856330623344, 'alis'),
 (0.0007065809792051378, 'alisave'),
 (0.005076417226266737, 'alison'),
 (0.011709436308839829, 'alistair'),
 (0.004559971450779515, 'aliu'),
 (0.017165915463983928, 'alium'),
 (0.01635189195892205, 'aliuum'),
 (0.11906648009332406, 'alive'),
 (-5.386406668399353e-05, 'alix'),
 (0.004916562884839947, 'alizarin'),
 (-0.011314303193720293, 'alka'),
 (0.02469712889168701, 'alkali'),
 (0.0016449660151115415, 'alkaline'),
 (0.0036186800261197973, 'alkaloid'),
 (-0.011314303193720293, 'alkaseltzer'),
 (0.0016449660151115415, 'alkene'),
 (0.3579669503474519, 'all'),
 (0.014941183607416674, 'allah'),
 (0.005905049605677353, 'allan'),
 (-0.0014318112715719646, 'alland'),
 (0.023225999807698832, 'allarity'),
 (0.01931621367950415, 'allawi'),
 (0.00856560948820523, 'alle'),
 (0.0153875508525

<div class="alert alert-success">
<b> Done !</b> Vous pouvez comparer les différentes méthodes de Data Mining et de Machine Learning sur les textes !
</div>

# Références

## Livres

Bengfort B., Bilbro B. et Ojeda T. (2018), "Applied Text Analysis with Python: Enabling Language-Aware Data Products with Machine Learning", Ed. O’Reilly.

Bird S., Klein E. et Loper E. (2009), "Natural Language Processing with Python - Analyzing Text with the Natural Language Toolkit", Ed. O'Reilly.

Bolukbasi T., Chang K., Zou J. Y., Saligrama V. et Kalai A. T. (2016), "Man is to
computer programmer as woman is to homemaker? debiasing word embeddings", Advances in Neural
Information Processing Systems, pp. 4349–4357. (disponible en ligne : https://arxiv.org/pdf/1607.06520.pdf)

Devlin J., Chang M., Lee K. et Toutanova K. (2019), "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding", Proceedings of the Conference of the North American Chapter of the Association for Computational Linguistics: Human Language Technologies, vol. 1, pp. 4171–4186. (disponible en ligne : https://www.aclweb.org/anthology/N19-1423/)

INRIA (2023), "Le numérique est-il un progrès durable ?", Pour La Science, vol. 546. (disponible en ligne : https://www.inria.fr/fr/numerique-progres-durable-environnement-pour-la-science)

Le Q. V. et Mikolov T. (2014),  "Distributed Representations of Sentences and Documents", Proceedings of the 31st International Conference on International Conference on Machine Learning, vol. 32 pp. 1188-1196. (disponible en ligne : https://arxiv.org/pdf/1405.4053.pdf)

Mikolov T., Chen K., Corrado G. et Dean J. (2013), "Efficient Estimation of Word Representations in Vector Space", 1301.3781, arXiv. (disponible en ligne : https://arxiv.org/pdf/1301.3781.pdf)

Ouyang L., Wu J., Jiang X., Almeida D., Wainwright C. L., Mishkin P., Zhang C., Agarwal S., Slama K., Ray A., Schulman J., Jacob Hilton J., Kelton F., Miller L., Simens M., Askell A., Welinder P., Christiano P., Leike J., Lowe R. (2022), "Training language models to follow instructions with human feedback". (disponible ici : https://arxiv.org/abs/2203.02155)

Peters M. E., Neumann M., Iyyer M., Gardner M., Clark C., Lee K. et Zettlemoyer L. (2018), "Deep contextualized word representations", Proceedings of NAACL. (disponible ici : https://arxiv.org/pdf/1802.05365.pdf)

Radford A., Narasimhan K., Salimans T., and Sutskever I. (2018), "Improving language understanding by generative pre-training". (disponible en ligne : https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)

Smith N. A. (2020), "Contextual Word Representations: Putting Words into Computers", Communications of the ACM, Vol. 63 No. 6, pp. 66-74. (disponible en ligne : https://arxiv.org/pdf/1902.06006.pdf)

Vajjala S., Majumder B., Gupta A. et Surana H. (2020), "Practical Natural Language Processing: A Comprehensive Guide to Building Real-World NLP Systems", Ed. O'Reilly.

Vaswani A., Shazeer N., Parmar N., Uszkoreit J., Jones L., Gomez A. N., Kaiser L. et Polosukhin I. (2017), "Attention Is All You Need", 31st Conference on Neural Information Processing Systems. (disponible en ligne : https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf)

Rong X. (2016), "word2vec Parameter Learning Explained". (disponible en ligne : https://arxiv.org/pdf/1411.2738.pdf)

Tunstall L., von Werra L., Wolf T. (2022), "Natural Language Processing with Transformers", Ed. O'Reilly.

## Sites Internet  (consultés en mars 2025) 

Allen Insitute for AI : https://allenai.org/

Documentation Python :
- librairie re : https://docs.python.org/3.7/library/re.html

NLTK : http://www.nltk.org/

fasttext : https://fasttext.cc/docs/en/english-vectors.html

Google Developers : https://developers.google.com/

GloVe : https://nlp.stanford.edu/projects/glove/

Hugging Face : https://huggingface.co/
Github : 
- BertViz https://github.com/jessevig/bertviz
- Gensim : https://github.com/RaRe-Technologies/gensim
- NLTK : https://github.com/nltk/nltk
- SpaCy : https://github.com/explosion/spaCy
- Scikit-Learn : https://github.com/scikit-learn/scikit-learn
- Stanza : https://github.com/stanfordnlp/stanza
- Textacy : https://github.com/chartbeat-labs/textacy
- Transformers  : https://github.com/huggingface

Gensim : https://radimrehurek.com/gensim/

Hugging Face : https://huggingface.co/



NLTK : http://www.nltk.org/

Kaggle : https://www.kaggle.com/

Scikit-Learn : https://scikit-learn.org/stable/ 

Université de Stanford (partie logiciels NLP) : https://nlp.stanford.edu/software/


SpaCy : https://spacy.io/

Stackoverflow :
- https://stackoverflow.com/a/5486535/12910854
- https://stackoverflow.com/a/45516258/12910854

Stanza : https://stanfordnlp.github.io/

Wikipedia :
- Expressions régulières : https://fr.wikipedia.org/wiki/Expression_r%C3%A9guli%C3%A8re
- Malédiction de la dimension : https://fr.wikipedia.org/wiki/Fl%C3%A9au_de_la_dimension
- TALN : https://fr.wikipedia.org/wiki/Traitement_automatique_des_langues
- Précision et recall : https://fr.wikipedia.org/wiki/Pr%C3%A9cision_et_rappel
- Grammaire non contextuelle : https://fr.wikipedia.org/wiki/Grammaire_non_contextuelle
- POS-tagging : https://en.wikipedia.org/wiki/Part-of-speech_tagging

Scikit-Learn : https://scikit-learn.org/stable/ 

Universal dependencies : https://universaldependencies.org/u/dep/

word2vec : https://code.google.com/archive/p/word2vec/